In [1]:
import tensorflow as tf
import os
import numpy as np
from constant import MODEL_DIR, PARTIAL_NAME

# Makes sure that for every run, we are getting the same results.
tf.random.set_seed(1234)

2026-11-14 01:01:49.319308: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-14 01:01:49.902732: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-11-14 01:01:51.747826: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
def save_model_safely(model, model_name, model_dir=MODEL_DIR):
    """Function that simply makes sure I do not overwrite a model.
    """
    print("Model Being Saved In Dir: ",MODEL_DIR)
    
    # checking to see if the directory path exist
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)
        
    # checking to make sure
    existing_models = os.listdir(model_dir)
    if model_name in existing_models:
        choice = input(f"'{model_name}' already exists in '{model_dir}'. Do you want to overwrite this model? (y/n): ").strip().lower()
        if choice != "y":
            print("Save cancelled!")
            return

    # see <https://www.tensorflow.org/guide/keras/serialization_and_saving>
    filepath = os.path.join(model_dir, model_name)
    model.save(filepath)
    print(f"Model Saved to {filepath}!")

In [3]:
# Checking what this file contains 
with np.load("notMNIST.npz", allow_pickle=True) as f:
    print("file:", f.files)

file: ['x_train', 'y_train', 'x_test', 'y_test']


In [4]:
print("--Get data--")
with np.load("notMNIST.npz", allow_pickle=True) as f:
    x_train, y_train = f['x_train'], f['y_train']
    x_test, y_test = f['x_test'], f['y_test']


--Get data--


In [5]:
print("--Process data--")
x_train, x_test = x_train / 255.0, x_test / 255.0

--Process data--


In [6]:
# I believe what we are doing is including a channel for
# tensorflow to showcase we are in grayscale, which is simply 
# the value of the pixel itself
print("Shape Before:", x_train.shape)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print("Shape After:", x_train.shape)
input_shape = (28, 28, 1)

Shape Before: (60000, 28, 28)
Shape After: (60000, 28, 28, 1)


In [7]:
print("--Make model--")
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
  tf.keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='sgd', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("--Fit model--")
model.fit(x_train, y_train, epochs=10, verbose=2)

--Make model--
--Fit model--


/usr/local/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-11-14 01:02:00.813898: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10
1875/1875 - 2s - 1ms/step - accuracy: 0.7817 - loss: 0.8381
Epoch 2/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8168 - loss: 0.6926
Epoch 3/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8224 - loss: 0.6721
Epoch 4/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8253 - loss: 0.6615
Epoch 5/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8267 - loss: 0.6546
Epoch 6/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8283 - loss: 0.6496
Epoch 7/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8292 - loss: 0.6457
Epoch 8/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8298 - loss: 0.6426
Epoch 9/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8306 - loss: 0.6400
Epoch 10/10
1875/1875 - 2s - 1ms/step - accuracy: 0.8310 - loss: 0.6377


In [8]:
print("--Evaluate model--")
model_loss1, model_acc1 = model.evaluate(x_train,  y_train, verbose=2)
model_loss2, model_acc2 = model.evaluate(x_test,  y_test, verbose=2)
print(f"Train / Test Accuracy: {model_acc1*100:.1f}% / {model_acc2*100:.1f}%")

--Evaluate model--
1875/1875 - 2s - 1ms/step - accuracy: 0.8324 - loss: 0.6331
313/313 - 0s - 2ms/step - accuracy: 0.8965 - loss: 0.4136
Train / Test Accuracy: 83.2% / 89.6%


In [9]:
save_model_safely(model,PARTIAL_NAME)

Model Being Saved In Dir:  model
Model Saved to model/notMNIST-Partial.keras!
